# Web Scraping con Playwright

Playwright es un *framework* de automatización de navegadores, alternativa moderna a Selenium. Su uso principal es la realización de pruebas en aplicaciones web, ya que se puede automatizar una serie de interacciones del usuario y comprobar que los resultados son los esperados. Sin embargo, por esa misma razón es un *framework* que permite hacer un *web scraping* muy preciso, simulando interacciones muy concretas de un usuario: rellenar formularios, pulsar botones y otros elementos, etc.

A diferencia de Selenium, Playwright **espera automáticamente** a que los elementos estén listos antes de interactuar con ellos (*auto-waiting*), lo que simplifica mucho el código en la mayoría de los casos.

Para utilizar Playwright en Python es necesario:
1. Instalar el paquete Python con `pip install playwright`
2. Descargar los navegadores que Playwright necesita (no hace falta buscar ni descargar manualmente ningún *driver*, Playwright se encarga de todo): `playwright install` (o `playwright install chromium` para instalar solo Chromium, por ejemplo)

La documentación de Playwright en Python está disponible en https://playwright.dev/python/

**Nota importante sobre Jupyter**: Playwright ofrece una API síncrona y otra asíncrona. La API síncrona **no funciona dentro de un notebook** porque Jupyter ya ejecuta su propio bucle de eventos `asyncio`; hay que usar la API asíncrona (con `async`/`await`).

In [1]:
!pip install playwright
!playwright install chromium
!playwright install firefox

## Selección de elementos

El proceso de automatizar la navegación de una página web consiste en seleccionar elementos de la página y realizar una acción con ellos: escribir texto, pulsar una tecla, hacer clic, etc. Playwright ofrece varias formas de seleccionar elementos mediante **locators**, siendo las más recomendadas las basadas en cómo un usuario percibiría la página (rol, texto, etiqueta), en vez de basarse en detalles internos de implementación como IDs o clases CSS:

* `page.get_by_label("...")`: busca un campo de formulario por su etiqueta asociada (`<label>`)
* `page.get_by_role("...", name="...")`: busca por rol de accesibilidad (`button`, `link`, `checkbox`...) y texto
* `page.get_by_text("...")`: busca un elemento por el texto que contiene
* `page.get_by_id("...")`: búsqueda por identificador (atributo `id`)
* `page.locator("...")`: selector CSS genérico (equivalente a `By.CSS_SELECTOR` en Selenium)
* `page.locator("xpath=...")`: búsqueda usando una ruta XPath

Ejemplos:

    elemento = page.get_by_label("Contraseña")
    elemento = page.locator("input#passwd-id")

Una vez encontrado un elemento (o mejor dicho, una vez creado el *locator*, ya que Playwright no lo busca hasta el momento de usarlo), se pueden hacer varias acciones. Por ejemplo:

    await elemento.fill("MADRID")
    await elemento.press("Enter")
    await elemento.click()

## Esperar a contenido asíncrono

En muchos casos las páginas web tienen contenido que se carga de manera asíncrona de distintas fuentes, y es necesario esperar a que esté disponible antes de comenzar a navegar la página web. A diferencia de Selenium, Playwright **espera automáticamente** a que un elemento exista y sea interactuable antes de actuar sobre él (hasta un tiempo máximo por defecto de 30 segundos), así que en la mayoría de casos no hace falta escribir ninguna espera explícita.

Si aun así se necesita esperar explícitamente a que aparezca un elemento concreto (por ejemplo, antes de leer el HTML de la página), se puede usar:

    await page.wait_for_selector("div.products")

En este caso se espera a que la página tenga cargado un elemento que encaje con el selector CSS `div.products`. Si pasado el tiempo máximo no aparece ningún elemento que encaje, se lanzará una excepción `TimeoutError`.

También se puede configurar un tiempo máximo de espera distinto por defecto para todas las operaciones de una página:

    page.set_default_timeout(12000)  # 12000 ms = 12 segundos

## Ejemplo completo con Playwright: obtener los productos de una tienda (página sintética)

In [2]:
from playwright.async_api import async_playwright

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        await page.goto("https://practice.qabrains.com/ecommerce")

        await page.get_by_label("email").fill("practice@qabrains.com")
        await page.get_by_label("password").fill("Password123")
        await page.get_by_role("button", name="Login").click()

        await page.wait_for_selector("div.products")

        html = await page.content()
        await browser.close()
        return html

html = await main()

### Nota: si esto se ejecutara como un script `.py` en vez de en un notebook

El código de la celda anterior tiene dos particularidades que existen **únicamente por estar dentro de un notebook Jupyter**, y que se podrían simplificar si el mismo código se guardara y ejecutara como un fichero `.py` normal:

1. **`await main()` a nivel superior no es válido fuera de un notebook.** Jupyter/IPython permite el *top-level await* como una facilidad especial del propio kernel (que ya tiene su bucle de eventos corriendo), pero en un script `.py` normal un `await` fuera de una función `async def` es directamente un **error de sintaxis**. Para ejecutar el mismo código como script habría que envolver la llamada con `asyncio.run(...)`:

    ```python
    import asyncio
    from playwright.async_api import async_playwright

    async def main():
        ...

    if __name__ == "__main__":
        html = asyncio.run(main())
    ```

2. **De hecho, en un script no hace falta usar `async`/`await` en absoluto.** La única razón para usar la API asíncrona de Playwright aquí es que la API síncrona no funciona dentro de un notebook (por el bucle de eventos ya en marcha de Jupyter). Fuera de un notebook esa limitación desaparece, así que en un script normal es más sencillo usar directamente la **API síncrona** de Playwright, sin ningún `async`/`await`:

    ```python
    from playwright.sync_api import sync_playwright

    def main():
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=False)
            page = browser.new_page()

            page.goto("https://practice.qabrains.com/ecommerce")
            page.get_by_label("email").fill("practice@qabrains.com")
            page.get_by_label("password").fill("Password123")
            page.get_by_role("button", name="Login").click()

            page.wait_for_selector("div.products")

            html = page.content()
            browser.close()
            return html

    if __name__ == "__main__":
        html = main()
    ```

En resumen: dentro de un notebook, la API asíncrona (con `async`/`await`) es la única opción viable. Fuera de un notebook (un script `.py`), la API síncrona suele ser la elección más simple, ya que evita `async`/`await` por completo sin perder ninguna funcionalidad.

In [3]:
print(html[:5000])

<!DOCTYPE html><html lang="en" style="scroll-behavior: smooth; --questrial-font: 'Questrial', 'Questrial Fallback'; --oswald-font: 'Oswald', 'Oswald Fallback'; color-scheme: light;" data-theme="light"><head><link rel="stylesheet" href="/_next/static/css/fc88df96f5c3b850.css" data-precedence="next"><link rel="preload" as="script" fetchpriority="low" href="/_next/static/chunks/webpack-7f89f43bd35b7caf.js"><script src="/_next/static/chunks/4bd1b696-f785427dddbba9fb.js" async=""></script><script src="/_next/static/chunks/1255-d271660abf95e684.js" async=""></script><script src="/_next/static/chunks/main-app-08607c612937c180.js" async=""></script><script src="/_next/static/chunks/1311-fcfcc5fecb199a25.js" async=""></script><script src="/_next/static/chunks/8720-d7f67a5726cf715d.js" async=""></script><script src="/_next/static/chunks/5125-8626032b05185706.js" async=""></script><script src="/_next/static/chunks/4243-e1cb11d2e59a0e94.js" async=""></script><script src="/_next/static/chunks/8571-

In [4]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(html, 'html.parser')

# DIV contenedor de todos los productos
productos = soup.find("div", "products")
print(len(productos))

9


In [5]:
productos.contents[0]  # Primer hijo

<div class="flex flex-col gap-3 relative group"><span class="absolute top-3 right-3 cursor-pointer" role="button"><button class="cursor-pointer" style="color: rgb(34, 34, 34);" type="button"><svg class="w-5 h-5" fill="currentColor" height="1em" stroke="currentColor" stroke-width="2" viewbox="0 0 512 512" width="1em" xmlns="http://www.w3.org/2000/svg"><path d="M349.6 64c-36.4 0-70.7 16.7-93.6 43.9C233.1 80.7 198.8 64 162.4 64 97.9 64 48 114.2 48 179.1c0 79.5 70.7 143.3 177.8 241.7L256 448l30.2-27.2C393.3 322.4 464 258.6 464 179.1 464 114.2 414.1 64 349.6 64zm-80.8 329.3l-4.2 3.9-8.6 7.8-8.6-7.8-4.2-3.9c-50.4-46.3-94-86.3-122.7-122-28-34.7-40.4-63.1-40.4-92.2 0-22.9 8.4-43.9 23.7-59.3 15.2-15.4 36-23.8 58.6-23.8 26.1 0 52 12.2 69.1 32.5l24.5 29.1 24.5-29.1c17.1-20.4 43-32.5 69.1-32.5 22.6 0 43.4 8.4 58.7 23.8 15.3 15.4 23.7 36.5 23.7 59.3 0 29-12.5 57.5-40.4 92.2-28.8 35.7-72.3 75.7-122.8 122z"></path></svg></button></span><a class="bg-gray-200 group-hover:bg-gray-300 p-4 decoration-0 mi

In [6]:
productos.contents[0].find('a', 'block')

<a class="text-lg block decoration-0 font-semibold font-oswald text-gray-900" href="/ecommerce/product-details?id=1">Sample Shirt Name</a>

In [7]:
productos.contents[0].find('span', 'text-lg')

<span class="text-lg font-bold text-black">$49.99</span>

In [8]:
def extrae_nombre_precio(div_elem):
    """ Extrae el nombre y precio (como float) de un DIV de un producto """
    nombre_a = div_elem.find("a", "block").string
    precio_str = div_elem.find('span', 'text-lg').string
    precio = float(precio_str[1:])
    return nombre_a, precio

# Lista de datos extraídos de productos, ordenador por precio descendente
prods = [extrae_nombre_precio(elem) for elem in productos]  # Recorro los hijos directos del DIV productos
sorted(prods, key = lambda x : x[1], reverse=True)

[('Sample Sunglass Name', 256.45),
 ('Sample Sunglass Name', 256.45),
 ('Sample Jacket Name', 129.5),
 ('Sample Jacket Name', 111.0),
 ('Sample Shoe Name', 89.0),
 ('Sample Trouser Name', 72.0),
 ('Sample T-Shirt Name', 56.45),
 ('Sample Shirt Name', 49.99),
 ('Sample T-Shirt Name', 45.0)]